# Vigilancia de métricas — corrida diaria

Cada métrica vigilada tiene su propia consulta, sus categorías y su grano. El motor le busca
anomalías, les pone gravedad, las agrupa en **eventos con identidad estable** y arma las
notificaciones.

Toda la configuración está en **`vig_oracle.py`** y todo el cálculo en **`vig_engine.py`**.

```
 ORIGEN (lector)                     MOTOR                        DESTINO (escritor)
 un SQL por vigilancia  ──>  series por grano (día/semana/mes)  ──>  VIG_SERIE      (con historia)
 [:desde, :hasta)            batería de detectores              ──>  VIG_EVENTO     (identidad estable)
 + historia guardada         gravedad + unificación             ──>  VIG_NOTIFICACION
                             atribución y causas                ──>  VIG_RESUMEN
                                        ^
                             VIG_CAUSA (la llenan ustedes)
```

### Nodo en el pipeline (Elyra / OpenShift AI)

| Propiedad | Valor |
|---|---|
| **File Dependencies** | `vig_oracle.py`, `vig_engine.py` |
| **Environment Variables** | `ORA_USER`, `ORA_PASSWORD`, `ORA_DSN`, `ORA_DEST_USER`, `ORA_DEST_PASSWORD`, `ORA_DEST_DSN` |
| **Opcionales** | `VG_FECHA_EJECUCION`, `VG_MODO`, `VG_DRY_RUN`, `VG_WEBHOOK_URL` |
| **Instalación** | `pip install pandas numpy oracledb` |

### Las cinco tablas

| Tabla | Qué guarda | Cómo se carga |
|---|---|---|
| `VIG_SERIE` | una fila por serie vigilada, con su historia completa en un CLOB | se reemplaza |
| `VIG_EVENTO` | una fila por evento, con id estable, estado y causa | se pisa lo recalculado, el resto queda |
| `VIG_CAUSA` | **la llenan ustedes**: qué causó cada cosa | el motor sólo la lee |
| `VIG_NOTIFICACION` | lo que hay que avisar, con el texto listo | se acumula |
| `VIG_RESUMEN` | conteo por vigilancia, grano y nivel de cada corrida | se acumula |

In [ ]:
# Parámetros (tag `parameters`)
import os

FECHA_EJECUCION = os.getenv("VG_FECHA_EJECUCION") or None   # "2026-09-15" para simular otro día
MODO = os.getenv("VG_MODO") or None                         # auto | completo | incremental
DRY_RUN = os.getenv("VG_DRY_RUN", "0") == "1"               # "1" = calcula y no escribe
print(f"FECHA_EJECUCION={FECHA_EJECUCION!r} MODO={MODO!r} DRY_RUN={DRY_RUN}")

## 1. Setup

In [ ]:
import sys, time
sys.path.insert(0, os.getcwd())

import pandas as pd
import vig_oracle as io
from vig_engine import VigEngine

log = io.configurar_logging("vigilancia")
cfg = io.build_config(fecha_ejecucion=FECHA_EJECUCION)
motor = VigEngine(cfg)
log.info("ejecución %s | último día incluido %s | %d vigilancias | notifica desde %s",
         motor.fechas.hoy.date(), motor.fechas.ayer.date(), len(cfg.vigilancias), cfg.nivel_notificacion)
t_inicio = time.time()

## 2. Tablas

Imprime el `CREATE TABLE` de las cinco (sin constraints, con la descripción de cada columna) y valida
que no falte ninguna columna **antes** de leer nada.

In [ ]:
print(io.ddl_sugerido(cfg))

with io.conexion_destino() as conn:
    io.validar_tablas(conn, cfg)

## 3. Plan, estado y causas

Decide desde cuándo leer la fuente. Si la historia guardada sirve, sólo relee los últimos
`DIAS_RELECTURA` días y el resto lo reconstruye del CLOB; si el pipeline no corrió en varios días,
relee desde el día siguiente a la última corrida para no dejar huecos.

También trae los eventos que quedaron abiertos (para no volver a notificar lo mismo) y las causas
que ustedes anotaron.

In [ ]:
with io.conexion_destino() as conn:
    desde, usa_estado, motivo = io.planificar(conn, cfg, modo=MODO or io.MODO_CORRIDA)
    estado = io.leer_estado(conn, cfg) if usa_estado else None
    abiertos = io.leer_eventos_abiertos(conn, cfg)
    causas = io.leer_causas(conn, cfg)
log.info("plan: lee la fuente desde %s (%s)", desde.date(), motivo)

## 4. Fuente y cálculo

Por cada vigilancia y grano: arma la serie, la pega con la historia guardada, corre los detectores,
arma los eventos, les pone gravedad y explica lo que puede.

In [ ]:
with io.conexion_origen() as conn:
    datos = io.leer_fuente(conn, cfg, desde)

res = motor.run(datos, estado=estado, eventos_previos=abiertos, causas=causas,
                desde_relectura=desde if usa_estado else None)
tablas = io.preparar_tablas(res, cfg)

## 5. Control

In [ ]:
io.resumen(motor, res)
res.resumen

In [ ]:
# Los eventos abiertos que más pesan, con quién los causó y si ya sabemos por qué.
abiertos_hoy = res.eventos[res.eventos.estado != "CERRADO"] if len(res.eventos) else res.eventos
(abiertos_hoy.sort_values("materialidad", ascending=False)
 .head(10)[["vigilancia", "grano", "clave", "detector", "nivel", "estado", "z", "materialidad",
            "atribucion", "causa", "historia_causas"]] if len(abiertos_hoy) else "sin eventos")

## 6. Notificar y guardar

`enviar()` escribe siempre en la tabla y además manda por los canales de `CANALES_NOTIFICACION`.
Si un canal falla, la corrida sigue y el error queda registrado en `BD_ESTADO_ENVIO`.

In [ ]:
tablas[io.TABLA_NOTIFICACION] = io.enviar(tablas[io.TABLA_NOTIFICACION])

if DRY_RUN:
    log.warning("DRY_RUN: no se escribe nada")
else:
    with io.conexion_destino() as conn:
        io.guardar(conn, tablas, cfg)
log.info("corrida OK en %.1fs", time.time() - t_inicio)

## Cómo funciona

### Los detectores

Cada uno busca una cosa distinta, y cada vigilancia elige cuáles usar:

| Detector | Qué encuentra |
|---|---|
| `hueco` | Venía con movimiento y se apagó: cero, o la categoría dejó de venir en la fuente. Es el que encuentra las cargas que fallaron. |
| `salto` | Pico o caída puntual contra la mediana móvil, medida con desvío robusto (MAD), que no se deja arrastrar por el propio pico. |
| `escalon` | El nivel cambió y se quedó ahí: compara las últimas mediciones contra la referencia. Pérdida real de negocio o cambio de criterio en la fuente. |
| `tendencia` | La pendiente de la última ventana se dio vuelta o se quebró contra la ventana anterior. |
| `estacional` | Compara contra el mismo período de otros años (o el mismo día de la semana), así diciembre no se compara con noviembre. |
| `racha` | Varios períodos seguidos del mismo lado de lo normal: avisa antes de que el escalón sea evidente. |
| `nueva` | Categoría que aparece por primera vez. |

En grano **día** no corre `estacional`: comparar contra "el mismo día de la semana" es más ruido que
señal. La estacionalidad se mira en semana y mes, contra el mismo período de otros años.

### Un problema, un evento

Un apagón lo marcan a la vez el hueco, el salto, el escalón y la racha. Si se notificaran los cuatro,
nadie leería las alertas. Se unifican: manda el detector más específico y los demás quedan en
`BD_CONFIRMAN`. En las pruebas, esto bajó de 17 notificaciones a 3 sobre los mismos datos.

### La gravedad mira tres cosas

1. **Cuán raro es**: desvío robusto (mediana y MAD), que no se deja arrastrar por el propio pico.
2. **Cuánto lleva así**: varios períodos seguidos suben un nivel. Ojo: sólo para los detectores de
   punto. En los que ya miran una ventana, "varios períodos" son ventanas que se pisan, no evidencia
   nueva.
3. **Cuánto pesa**: una serie que pesa menos de `PESO_RELATIVO_MINIMO` veces la serie promedio no
   puede ser CRITICO, y tampoco pasa nada por debajo de `materialidad_minima`. El peso se mide contra
   la serie promedio y no contra el total: con 3 series o con 3.000, "chica" significa lo mismo.

### Ruido: escala, estacionalidad y pisos

Tres cosas evitan la inundación de alertas, y cada una salió de medirla:

- **Escala logarítmica** cuando la métrica es positiva: el ruido de una venta es multiplicativo
  (±20%), no de tantos USD. Medirlo en línea recta infla las colas.
- **Desestacionalización semanal** en grano día: cada día se compara contra los de su mismo día.
  Sin esto, "siete días seguidos por encima de lo normal" pasa todas las semanas.
- **Pisos**: `DESVIO_RELATIVO_MINIMO` (una diferencia del 2% no es noticia aunque sea estadísticamente
  rarísima) y `PISO_SIGMA_RELATIVO` (los totales mensuales son tan parejos que sin piso cualquier
  diferencia da un desvío enorme).

Con 2.000 series simuladas y 20 fallas reales inyectadas: **las 20 detectadas**, 33 notificaciones en
total y el 10% de las series sanas con algún evento abierto, casi todos de nivel ATENCION que no
notifica. Los umbrales se terminan de calibrar con sus datos: `VIG_RESUMEN` muestra el volumen.

### Identidad del evento

El mismo problema conserva su `BD_ID_EVENTO` mientras siga abierto: no se re-notifica todos los días.
Se vuelve a notificar **sólo si empeora de nivel**. Cuando la métrica se normaliza, el evento se
cierra solo con su `FECHA_CIERRE`, y queda la duración real.

### Causas: lo que el motor deduce y lo que ustedes saben

- **`BD_ATRIBUCION`** la calcula el motor: compara el tramo del evento contra el tramo anterior y
  ordena a los hijos (clientes, productos) por cuánto aportaron al cambio. Ejemplo real de la prueba:
  *"5 de 20 explican el 53% del cambio: C1 (-2.071 USD), C0 (-1.836 USD)..."*.
- **`BD_CAUSA`** sale de `VIG_CAUSA`, la tabla que llenan ustedes. Si la fecha del evento cae dentro
  del rango anotado, se pega sola.
- **`BD_HISTORIA_CAUSAS`** es la memoria: *"ya pasó 1 vez; la última el 2025-09-15: cierre por
  inventario"*. Eso es lo que evita investigar dos veces lo mismo.

### Siempre queda algo guardado

Toda serie vigilada tiene su fila en `VIG_SERIE` con su historia completa, tenga o no eventos, y sin
exigir un mínimo de historia. Una serie que arranca hoy genera un evento `nueva` de nivel INFO: no
molesta a nadie, pero queda el registro.